# **Solving Sudoku 4x4 Problems: GBFS, Hill Climbing, dan DFS**
Notebook ini berisikan code untuk menyelesaikan permasalahan Sudoku 4x4 dengan metode GBFS, Hill Climbing dan DFS

## **Datasets and Function**
Cell berikut berisikan setup datasets beserta function yang dibutuhkan dalam pencarian

In [ ]:
import heapq
import copy
import itertools
import random
import pandas as pd


df = pd.read_csv("4x4_sudoku_unique_solution.csv", sep='\t', dtype="string")
sudoku_dataset = df["Puzzle"].to_list()


def string_to_board(state_str):
    return [[int(state_str[r * 4 + c]) for c in range(4)] for r in range(4)]

chosen_string = random.choice(sudoku_dataset)
initial_board = string_to_board(chosen_string)


def print_board(board):
    for i, row in enumerate(board):
        if i > 0 and i % 2 == 0:
            print("-" * 13)
        print(" ".join(str(cell) if cell != 0 else "." for cell in row))
    print()



def is_valid(board, row, col, num):
    if num in board[row]:
        return False
    if num in [board[r][col] for r in range(4)]:
        return False
    start_row, start_col = (row // 2) * 2, (col // 2) * 2
    for r in range(start_row, start_row + 2):
        for c in range(start_col, start_col + 2):
            if board[r][c] == num:
                return False
    return True

def get_empty_cell(board):
    for r in range(4):
        for c in range(4):
            if board[r][c] == 0:
                return (r, c)
    return None

def heuristic(board):
    total_value = 0
    for r in range(4):
        for c in range(4):
            if board[r][c] == 0:
                valid_options = sum(1 for num in range(1, 5) if is_valid(board, r, c, num))
                if valid_options == 0:
                    return float('inf')
                total_value += valid_options
    return total_value

## **Greedy Best-First Search (GBFS)**

In [ ]:
import time
import tracemalloc



def gbfs_sudoku(board):
    counter = itertools.count()
    pq = []
    h_init = heuristic(board)

    if h_init == float('inf'):
        print("Initial state tidak valid (memiliki jalur buntu dari awal).")
        return None

    heapq.heappush(pq, (h_init, next(counter), board))
    visited = set()
    steps = 0

    print("State awal:")
    print_board(board)
    print(f"h awal (SRV): {h_init}\n")

    while pq:
        h, _, current_board = heapq.heappop(pq)

        if h == float('inf'):
            continue

        steps += 1

        print(f"Iterasi {steps} | h: {h}")
        print_board(current_board)

        if get_empty_cell(current_board) is None:
            print(f"Solusi ditemukan dalam {steps} iterasi langkah pencarian!")
            return current_board

        empty = get_empty_cell(current_board)
        if not empty:
            continue

        row, col = empty

        board_tuple = tuple(tuple(r) for r in current_board)
        if board_tuple in visited:
            continue
        visited.add(board_tuple)

        for num in range(1, 5):
            if is_valid(current_board, row, col, num):
                new_board = copy.deepcopy(current_board)
                new_board[row][col] = num

                h_new = heuristic(new_board)

                if h_new != float('inf'):
                    heapq.heappush(pq, (h_new, next(counter), new_board))

    print("Tidak ditemukan solusi.")
    return None

gbfs_sudoku(initial_board)

State awal:
. 3 . 1
2 . . .
-------------
. 2 . 3
. 4 1 .

h awal (SRV): 11

Iterasi 1 | h: 11
. 3 . 1
2 . . .
-------------
. 2 . 3
. 4 1 .

Iterasi 2 | h: 9
4 3 . 1
2 . . .
-------------
. 2 . 3
. 4 1 .

Iterasi 3 | h: 8
4 3 2 1
2 . . .
-------------
. 2 . 3
. 4 1 .

Iterasi 4 | h: 7
4 3 2 1
2 1 . .
-------------
. 2 . 3
. 4 1 .

Iterasi 5 | h: 5
4 3 2 1
2 1 3 .
-------------
. 2 . 3
. 4 1 .

Iterasi 6 | h: 4
4 3 2 1
2 1 3 4
-------------
. 2 . 3
. 4 1 .

Iterasi 7 | h: 3
4 3 2 1
2 1 3 4
-------------
1 2 . 3
. 4 1 .

Iterasi 8 | h: 2
4 3 2 1
2 1 3 4
-------------
1 2 4 3
. 4 1 .

Iterasi 9 | h: 1
4 3 2 1
2 1 3 4
-------------
1 2 4 3
3 4 1 .

Iterasi 10 | h: 0
4 3 2 1
2 1 3 4
-------------
1 2 4 3
3 4 1 2

Solusi ditemukan dalam 10 iterasi langkah pencarian!


[[4, 3, 2, 1], [2, 1, 3, 4], [1, 2, 4, 3], [3, 4, 1, 2]]

## **Hill Climbing**

In [7]:
def get_neighbors(board):
    neighbors = []
    for r in range(4):
        for c in range(4):
            if board[r][c] == 0:
                for num in range(1, 5):
                    if is_valid(board, r, c, num):
                        new_board = copy.deepcopy(board)
                        new_board[r][c] = num
                        neighbors.append(new_board)
    return neighbors

def hill_climbing_sudoku(board, verbose=True):
    current = copy.deepcopy(board)
    current_h = srv(current)
    steps = 0

    if verbose:
        print("State awal:")
        print_board(current)
        print(f"Jumlah sel kosong awal: {current_h}\n")

    while current_h > 0:
        neighbors = get_neighbors(current)
        if not neighbors:
            break

        scored = [(srv(n), n) for n in neighbors]
        best_h = min(h for h, _ in scored)
        if best_h >= current_h:
            break

        best_neighbors = [n for h, n in scored if h == best_h]
        current = random.choice(best_neighbors)
        current_h = best_h
        steps += 1

        if verbose:
            print(f"Langkah {steps} | h: {current_h}")
            print_board(current)

    if verbose:
        if current_h == 0:
            print(f"Solusi ditemukan dalam {steps} langkah!")
        else:
            print(f"Terjebak di local optimum setelah {steps} langkah (h: {current_h}).")

    return current

hill_climbing_sudoku(initial_board)

State awal:
. 3 . 1
2 . . .
-------------
. 2 . 3
. 4 1 .

Jumlah sel kosong awal: 11

Langkah 1 | h: 8
. 3 . 1
2 . . .
-------------
. 2 4 3
. 4 1 .

Langkah 2 | h: 7
. 3 . 1
2 . . .
-------------
. 2 4 3
. 4 1 2

Langkah 3 | h: 6
. 3 . 1
2 . . .
-------------
. 2 4 3
3 4 1 2

Langkah 4 | h: 5
4 3 . 1
2 . . .
-------------
. 2 4 3
3 4 1 2

Langkah 5 | h: 4
4 3 . 1
2 . . .
-------------
1 2 4 3
3 4 1 2

Langkah 6 | h: 3
4 3 2 1
2 . . .
-------------
1 2 4 3
3 4 1 2

Langkah 7 | h: 2
4 3 2 1
2 . 3 .
-------------
1 2 4 3
3 4 1 2

Langkah 8 | h: 1
4 3 2 1
2 1 3 .
-------------
1 2 4 3
3 4 1 2

Langkah 9 | h: 0
4 3 2 1
2 1 3 4
-------------
1 2 4 3
3 4 1 2

Solusi ditemukan dalam 9 langkah!


[[4, 3, 2, 1], [2, 1, 3, 4], [1, 2, 4, 3], [3, 4, 1, 2]]

## **Depth First Search (DFS)**

In [8]:
def is_valid_start(board):
    for r in range(4):
        for c in range(4):
            num = board[r][c]
            if num != 0:
                board[r][c] = 0
                ok = is_valid(board, r, c, num)
                board[r][c] = num
                if not ok:
                    return False
    return True

def solve_dfs(board, state, verbose):
    empty = get_empty_cell(board)
    if empty is None:
        return True

    row, col = empty
    for num in range(1, 5):
        if not is_valid(board, row, col, num):
            continue

        board[row][col] = num
        state["steps"] += 1
        if verbose:
            print(f"Langkah {state['steps']}: Taruh {num} di ({row}, {col})")
            print_board(board)

        if solve_dfs(board, state, verbose):
            return True

        board[row][col] = 0
        state["steps"] += 1
        if verbose:
            print(f"Langkah {state['steps']}: Backtrack di ({row}, {col})")
            print_board(board)

    return False

def dfs_sudoku(board, verbose=True):
    current = copy.deepcopy(board)

    if not is_valid_start(current):
        if verbose:
            print("Angka awal sudah saling bentrok, puzzle tidak punya solusi.")
        return None

    state = {"steps": 0}
    solved = solve_dfs(current, state, verbose)

    if verbose:
        if solved:
            print(f"Solusi ditemukan dalam {state['steps']} langkah (termasuk backtrack).")
        else:
            print(f"Tidak ada solusi. Langkah: {state['steps']}.")

    return current if solved else None

dfs_sudoku(initial_board)

Langkah 1: Taruh 4 di (0, 0)
4 3 . 1
2 . . .
-------------
. 2 . 3
. 4 1 .

Langkah 2: Taruh 2 di (0, 2)
4 3 2 1
2 . . .
-------------
. 2 . 3
. 4 1 .

Langkah 3: Taruh 1 di (1, 1)
4 3 2 1
2 1 . .
-------------
. 2 . 3
. 4 1 .

Langkah 4: Taruh 3 di (1, 2)
4 3 2 1
2 1 3 .
-------------
. 2 . 3
. 4 1 .

Langkah 5: Taruh 4 di (1, 3)
4 3 2 1
2 1 3 4
-------------
. 2 . 3
. 4 1 .

Langkah 6: Taruh 1 di (2, 0)
4 3 2 1
2 1 3 4
-------------
1 2 . 3
. 4 1 .

Langkah 7: Taruh 4 di (2, 2)
4 3 2 1
2 1 3 4
-------------
1 2 4 3
. 4 1 .

Langkah 8: Taruh 3 di (3, 0)
4 3 2 1
2 1 3 4
-------------
1 2 4 3
3 4 1 .

Langkah 9: Taruh 2 di (3, 3)
4 3 2 1
2 1 3 4
-------------
1 2 4 3
3 4 1 2

Solusi ditemukan dalam 9 langkah (termasuk backtrack).


[[4, 3, 2, 1], [2, 1, 3, 4], [1, 2, 4, 3], [3, 4, 1, 2]]

## **Analisis Perbandingan Algoritma**

In [ ]:
import time
import tracemalloc
from contextlib import redirect_stdout

class NullWriter:
    def write(self, text):
        return len(text)

    def flush(self):
        pass

def run_silent(search_fn, board):
    with redirect_stdout(NullWriter()):
        return search_fn(board)

def measure_once(search_fn, board):
    start_empty = heuristic(board)

    board_time = copy.deepcopy(board)
    start = time.perf_counter()
    result = run_silent(search_fn, board_time)
    elapsed_ms = (time.perf_counter() - start) * 1000

    board_mem = copy.deepcopy(board)
    tracemalloc.start()
    run_silent(search_fn, board_mem)
    _, peak_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    if result is None:
        return elapsed_ms, peak_bytes / 1024, False, None

    final_empty = heuristic(result)
    return elapsed_ms, peak_bytes / 1024, final_empty == 0, start_empty - final_empty

def analyze_algorithms(algorithms, puzzles, runs=1):
    rows = []
    total = len(puzzles) * runs

    for name, search_fn in algorithms.items():
        results = []
        for _, board in puzzles:
            for _ in range(runs):
                results.append(measure_once(search_fn, board))

        avg_time = sum(r[0] for r in results) / total
        avg_mem = sum(r[1] for r in results) / total
        solved_count = sum(1 for r in results if r[2])
        rows.append([
            name,
            f"{solved_count}/{total}",
            f"{avg_time:.3f}",
            f"{avg_mem:.2f}",
        ])

    headers = ["Algoritma", "Berhasil", "Waktu (ms)", "Memori Puncak (KB)"]
    widths = [max(len(str(x)) for x in col) for col in zip(headers, *rows)]
    separator = "+" + "+".join("-" * (w + 2) for w in widths) + "+"

    def format_row(row):
        return "| " + " | ".join(str(v).ljust(w) for v, w in zip(row, widths)) + " |"

    print(separator)
    print(format_row(headers))
    print(separator)
    for row in rows:
        print(format_row(row))
    print(separator)

algorithms = {
    "GBFS": gbfs_sudoku,
    "Hill Climbing 1": lambda b: hill_climbing_sudoku(b, verbose=False),
    "Hill Climbing 2": lambda b: hill_climbing_sudoku(b, verbose=False),
    "Hill Climbing 3": lambda b: hill_climbing_sudoku(b, verbose=False),
    "Hill Climbing 4": lambda b: hill_climbing_sudoku(b, verbose=False),
    "Hill Climbing 5": lambda b: hill_climbing_sudoku(b, verbose=False),
    "DFS": lambda b: dfs_sudoku(b, verbose=False)
}
puzzles = [(state_str, string_to_board(state_str)) for state_str in sudoku_dataset]
print(f"Total initial state: {len(puzzles)}")
analyze_algorithms(algorithms, puzzles, runs=1)

Total initial state: 288


NameError: name 'heuristic' is not defined